# Decoder inputs and labels for a batch

Hugging Face's `T5ForConditionalGeneration` takes the targets as `labels=` and does two
jobs with them: it builds the decoder's input, and it computes the loss. Here you do both
in NumPy, for a batch of three targets of different lengths.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. Each row of `labels` is one target, padded with `IGNORE`, the value
the loss skips.

In [ ]:
import numpy as np

PAD = 0        # T5's padding token, which is also its start token
END = 1        # the end token that closes every target
IGNORE = -100  # a label value the loss skips
V = 150        # vocabulary size

# Three tokenized targets of different lengths, each closed by the end token.
targets = [[37, 52, 9, END], [88, END], [14, 61, 23, 5, END]]
T = max(len(t) for t in targets)
labels = np.full((len(targets), T), IGNORE)
for i, t in enumerate(targets):
    labels[i, : len(t)] = t
print(labels)

## Part 1 (coding task): The decoder's input

Write `decoder_inputs(labels)`. It returns an integer array of the same shape as
`labels`, whose row $i$ is the decoder's input for target $i$: the start token `PAD`,
then the labels of row $i$ moved one place to the right, with the last one dropped.
`IGNORE` is not a token the embedding table knows, so every `IGNORE` that ends up in the
decoder's input becomes `PAD`.

`labels` must come out of the call unchanged: the loss in Part 2 still needs it.

In [ ]:
def decoder_inputs(labels):
    """The decoder's input for each row of labels: PAD, then the labels moved right."""
    # TODO
    raise NotImplementedError

In [ ]:
before = labels.copy()
dec_in = decoder_inputs(labels)
print(dec_in)
assert np.array_equal(labels, before), "decoder_inputs changed labels: build a new array"
assert dec_in.shape == labels.shape, f"expected shape {labels.shape}, got {dec_in.shape}: drop the last column"
assert np.issubdtype(dec_in.dtype, np.integer), "the decoder's input should be integer token ids"
left = np.array([[52, 9, 1, 0, 0], [1, 0, 0, 0, 0], [61, 23, 5, 1, 0]])
assert not np.array_equal(dec_in, left), "the labels moved left: move them right, behind the start token"
assert (dec_in[:, 0] == PAD).all(), "every row should start with the start token, PAD"
assert (dec_in >= 0).all(), "the decoder's input still holds IGNORE (-100): turn it into PAD"
expected = np.array([[0, 37, 52, 9, 1], [0, 88, 1, 0, 0], [0, 14, 61, 23, 5]])
assert np.array_equal(dec_in, expected), f"expected\n{expected}"
single = decoder_inputs(np.array([[7, END]]))
assert np.array_equal(single, [[PAD, 7]]), f"for the labels [[7, 1]] expected [[0, 7]], got {single.tolist()}"
print("Part 1 passed")

## Part 2 (coding task): The loss

Write `seq2seq_loss(logits, labels)`. `logits` has shape `(batch, T, V)`: the decoder's
scores for the next token at every position. Return the mean cross-entropy over the
positions whose label is not `IGNORE`, as `T5ForConditionalGeneration` reports it, so
that every real label counts once, whichever row it is in. Large logits must not
overflow.

In [ ]:
def seq2seq_loss(logits, labels):
    """Mean cross-entropy over the positions whose label is not IGNORE."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(5)
logits = 2.0 * rng.standard_normal((3, T, V))
loss = seq2seq_loss(logits, labels)
print(loss)
assert np.ndim(loss) == 0, "return a single number"
assert not np.isclose(loss, 7.529327, atol=1e-5), "the IGNORE positions were averaged in: leave them out"
assert not np.isclose(loss, 5.427805, atol=1e-5), "the sum is right, but it is divided by every position: divide by the number of real labels"
assert not np.isclose(loss, 7.709934, atol=1e-5), "each row was averaged first, so a short row counts as much as a long one: average over all real labels at once"
assert np.isclose(loss, 7.401552, atol=1e-5), "the loss is wrong: it is minus the mean log-probability of each real label"
assert np.isclose(seq2seq_loss(np.zeros((3, T, V)), labels), np.log(V)), "with every logit equal, the loss should be ln V"
assert np.isfinite(seq2seq_loss(1000 * logits, labels)), "the loss overflowed for large logits: subtract each position's largest logit first"
print("Part 2 passed")

## Solution

### Part 1: The decoder's input

In [ ]:
def decoder_inputs(labels):
    """The decoder's input for each row of labels: PAD, then the labels moved right."""
    dec_in = np.full_like(labels, PAD)   # a new array, so labels is never written to
    dec_in[:, 1:] = labels[:, :-1]       # shift right, dropping the last label
    dec_in[dec_in == IGNORE] = PAD
    return dec_in

**Why it works.** `np.full_like(labels, PAD)` makes a new array, so `labels` is never
written to, and its first column already holds the start token. Copying `labels[:, :-1]`
into columns 1 onward moves every row right by one and drops the last label, which no
position needs as input, since nothing comes after it. Every `IGNORE` copied across then
becomes `PAD`. In the second row the end token lands in column 2 and the columns after it
read pads, but their labels are `IGNORE`, so the loss never looks at what they predict.

### Part 2: The loss

In [ ]:
def seq2seq_loss(logits, labels):
    """Mean cross-entropy over the positions whose label is not IGNORE."""
    top = logits.max(axis=-1, keepdims=True)                   # every exponent below is <= 0
    logp = logits - top - np.log(np.sum(np.exp(logits - top), axis=-1, keepdims=True))
    rows, cols = np.nonzero(labels != IGNORE)                  # the real labels only
    return -logp[rows, cols, labels[rows, cols]].mean()

**Why it works.** Subtracting each position's largest logit leaves the softmax unchanged
and keeps every exponent at or below 0, so nothing overflows: with the logits multiplied
by 1000, a plain `np.exp` returns `inf`. `np.nonzero` lists the (row, position) pairs of
the real labels, and only those pick out log-probabilities. That matters because
`IGNORE` is a valid NumPy index: `logp[..., -100]` quietly reads entry 50 of the
150-token vocabulary, so averaging over every position gives 7.53 instead of 7.40, with
no error. One mean over all 11 real labels weights every token the same, as
`T5ForConditionalGeneration` does. Averaging each row first would give the two-token
target as much weight as the five-token one.